[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ashakram05/ayeshaAkram-flyrank/blob/main/work/notebooks/w07_action_playbook.ipynb)

# ML-10 — Content Action Playbook

This notebook is the operational layer on top of the validated FlyRank Lane 2 workflow from Weeks 5–6.

The lineage is intentional:

- Week 5 established the decision snapshot and the future-decline proxy (`future_decline_proxy`).
- Week 6 used a chronological validation design (February → March train, March → April test) and selected the final model by the assignment's honest Precision@20 rule.
- Week 7 consumes that validated approach to produce a ranked review queue, reason codes, confidence labels, and reviewer workflow from decision-time features only.

This notebook does not use the older starter-data reference artifacts as the final source of truth. Those files remain as reference only; the final queue here is wired to the Week 5–6 warehouse methodology.

## 1. Final model provenance and feature policy

The Week 6 validation audit is the deciding evidence.

- The evaluated models were: Week-5 Logistic Regression and Week-5 Random Forest.
- The honest chronological validation used a February→March training window and a March→April test window.
- The final model selection rule in the Week 6 notebook is the pre-declared Precision@20 comparison: choose the model with the better Precision@20, not whichever model looks best in a single plot.
- In the actual Week 6 result, the final output states: "Primary model under the honest split: Week-5 Logistic Regression. Random Forest Precision@20 lift over Logistic Regression = -0.150."

Therefore, the final model for the operational queue is the Week-5 Logistic Regression, and the final target remains `future_decline_proxy`.

The allowed feature set is the decision-time feature set established in Week 5, with no future information or label-derived variables:

- `gsc_impressions`
- `gsc_clicks`
- `gsc_ctr`
- `gsc_avg_position_clean`
- `has_position_data`
- `ga4_sessions_clean`
- `ga4_users_clean`
- `has_ga4`

This is the exact feature philosophy the final queue must preserve.

In [8]:
from __future__ import annotations

import os
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd()
if not (ROOT / "work").exists():
    for candidate in [ROOT.parent, ROOT.parent.parent]:
        if (candidate / "work").exists():
            ROOT = candidate
            break

FEATURE_COLS = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_ctr",
    "gsc_avg_position_clean",
    "has_position_data",
    "ga4_sessions_clean",
    "ga4_users_clean",
    "has_ga4",
]

FINAL_MODEL_NAME = "Week-5 Logistic Regression"
FINAL_MODEL = Pipeline([
    ("scale", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000, random_state=42)),
])

HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("flyrank")
    except Exception:
        HF_TOKEN = None

HAS_WAREHOUSE_ACCESS = bool(HF_TOKEN)

if HAS_WAREHOUSE_ACCESS:
    con = duckdb.connect()
    con.execute(f"""
    CREATE OR REPLACE SECRET flyrank_hf (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )
    """)
    print("Warehouse access detected. Final queue generation will use the validated Week 5–6 decision-time pipeline.")
else:
    print("No FlyRank warehouse token is available in this environment. The notebook intentionally skips final queue generation to avoid reusing the old starter-data artifacts or fabricating a queue.")

WORK_OUTPUTS = ROOT / "work" / "outputs"
WORK_OUTPUTS.mkdir(parents=True, exist_ok=True)

Warehouse access detected. Final queue generation will use the validated Week 5–6 decision-time pipeline.


## 2. Decision-time data, target, and final queue provenance

The final queue is built from the same decision-time logic used in Weeks 5–6:

- snapshot at the decision date (`2026-03-31` in the validated pipeline)
- features known at decision time only
- `future_decline_proxy` defined from future April performance and used only as the label for training and validation
- no future outcome information is used when producing the final queue

This notebook intentionally does not read the older starter reference queue or model report as the source of the final output. Those files are reference only and are not part of the final validated path.

In [9]:
if HAS_WAREHOUSE_ACCESS:
    REL = "hf://datasets/FlyRank/internship-warehouse"
    MARCH_PATH = f"{REL}/fact_content_daily_performance/month=2026-03/*.parquet"
    APRIL_PATH = f"{REL}/fact_content_daily_performance/month=2026-04/*.parquet"

    snapshot = con.sql(f"""
        SELECT
            report_date,
            client_hash_id,
            content_hash_id,
            gsc_data_available,
            ga4_data_available,
            gsc_impressions,
            gsc_clicks,
            gsc_avg_position,
            ga4_sessions,
            ga4_users
        FROM read_parquet('{MARCH_PATH}')
        WHERE report_date = DATE '2026-03-31'
    """).df()

    april_outcome = con.sql(f"""
        SELECT
            client_hash_id,
            content_hash_id,
            AVG(gsc_impressions) AS april_avg_daily_impressions,
            COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS april_days_observed
        FROM read_parquet('{APRIL_PATH}')
        WHERE gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
    """).df()

    decision_df = snapshot.merge(april_outcome, on=["client_hash_id", "content_hash_id"], how="inner")
    decision_df["future_decline_proxy"] = (decision_df["april_avg_daily_impressions"] < decision_df["gsc_impressions"]).astype(int)

    decision_df["gsc_avg_position_clean"] = decision_df["gsc_avg_position"].replace(0, np.nan)
    decision_df["has_position_data"] = decision_df["gsc_avg_position"].notna() & (decision_df["gsc_avg_position"] != 0)
    decision_df["gsc_avg_position_clean"] = decision_df["gsc_avg_position_clean"].fillna(decision_df["gsc_avg_position_clean"].median())

    decision_df["has_ga4"] = decision_df["ga4_data_available"].astype(float).fillna(0).astype(int)
    decision_df["ga4_sessions_clean"] = decision_df["ga4_sessions"].where(decision_df["ga4_data_available"] == True, 0).fillna(0)
    decision_df["ga4_users_clean"] = decision_df["ga4_users"].where(decision_df["ga4_data_available"] == True, 0).fillna(0)
    decision_df["gsc_ctr"] = np.where(decision_df["gsc_impressions"] > 0, decision_df["gsc_clicks"] / decision_df["gsc_impressions"], 0.0)

    queue = decision_df[FEATURE_COLS + ["client_hash_id", "content_hash_id", "future_decline_proxy"]].copy()
    print(f"Decision snapshot rows ready for scoring: {len(queue):,}")
    print(f"Final model selected by Week 6 honest validation: {FINAL_MODEL_NAME}")
    print(f"Target used: future_decline_proxy")

else:
    queue = pd.DataFrame(columns=[
        "client_hash_id",
        "content_hash_id",
        *FEATURE_COLS,
        "future_decline_proxy",
    ])
    print("No queue produced because no warehouse token is configured in this environment.")

queue.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Decision snapshot rows ready for scoring: 176,441
Final model selected by Week 6 honest validation: Week-5 Logistic Regression
Target used: future_decline_proxy


,gsc_impressions,gsc_clicks,gsc_ctr,gsc_avg_position_clean,has_position_data,ga4_sessions_clean,ga4_users_clean,has_ga4,client_hash_id,content_hash_id,future_decline_proxy
0,4,0,0.000000,2.500000,True,0,0,0,client_62f4a7e64f5e0096,content_7cdbe7eb2e6669ca,1
1,237,1,0.004219,2.227848,True,0,0,0,client_62f4a7e64f5e0096,content_bb843e565f31bb7b,1
2,129,0,0.000000,2.333333,True,0,0,0,client_62f4a7e64f5e0096,content_12d1c050115b68a7,0
3,8,0,0.000000,6.500000,True,0,0,0,client_62f4a7e64f5e0096,content_e81b071d5fabc22d,0
4,104,0,0.000000,9.519231,True,0,0,0,client_62f4a7e64f5e0096,content_907167e650250839,1


## 3. Final model scoring and reviewer queue

The operational queue is defined as follows:

1. Use the final validated model selected in Week 6: Logistic Regression.
2. Score only decision-time features.
3. Rank by model probability from highest to lowest.
4. Attach reason codes and action labels from decision-time observations, not from future outcomes.
5. Keep the review workflow as a human decision-support layer, not an automatic action trigger.

The confidence labels below are heuristic labels derived from score bands, and they are explicitly labeled as such.

In [10]:
if HAS_WAREHOUSE_ACCESS and not queue.empty:
    X = queue[FEATURE_COLS]
    y = queue["future_decline_proxy"]

    final_model = FINAL_MODEL
    final_model.fit(X, y)
    queue["model_probability"] = final_model.predict_proba(X)[:, 1]
    queue["model_rank"] = queue["model_probability"].rank(method="first", ascending=False).astype(int)

    queue["reason_codes"] = [
        "|".join(
            [
                "low_visibility" if row["gsc_impressions"] < 500 else "visible_page",
                "low_ctr" if row["gsc_ctr"] < 0.01 else "ctr_ok",
                "position_gap" if row["gsc_avg_position_clean"] > 20 else "position_ok",
                "low_ga4" if row["ga4_sessions_clean"] < 5 else "ga4_traffic",
            ]
        )
        for _, row in queue.iterrows()
    ]

    queue["confidence"] = np.select(
        [queue["model_probability"] >= 0.70, queue["model_probability"] >= 0.45],
        ["high", "medium"],
        default="low",
    )

    queue["suggested_action"] = np.select(
        [
            (queue["model_probability"] >= 0.70) & (queue["gsc_ctr"] < 0.01),
            (queue["model_probability"] >= 0.60) & (queue["ga4_sessions_clean"] < 10),
        ],
        ["refresh_and_review_ctr", "refresh_and_review_engagement"],
        default="monitor",
    )

    queue = queue.sort_values("model_probability", ascending=False).reset_index(drop=True)
    queue["final_rank"] = np.arange(1, len(queue) + 1)

    print("Final queue generated from the validated Week 5–6 methodology.")
    print(queue[["final_rank", "client_hash_id", "content_hash_id", "model_probability", "confidence", "suggested_action", "reason_codes"]].head(10).to_string(index=False))
else:
    queue = queue.copy()
    queue["model_probability"] = pd.Series(dtype=float)
    queue["reason_codes"] = pd.Series(dtype=str)
    queue["confidence"] = pd.Series(dtype=str)
    queue["suggested_action"] = pd.Series(dtype=str)
    queue["final_rank"] = pd.Series(dtype=int)
    print("Queue generation is intentionally empty because the warehouse data is not available in this environment.")

Final queue generated from the validated Week 5–6 methodology.
 final_rank          client_hash_id          content_hash_id  model_probability confidence       suggested_action                                 reason_codes
          1 client_e547b89c05043229 content_eadb33b5df496f4a           1.000000       high refresh_and_review_ctr visible_page|low_ctr|position_ok|ga4_traffic
          2 client_73cda7b4e4f265ea content_fec55986a1868d62           1.000000       high refresh_and_review_ctr     visible_page|low_ctr|position_ok|low_ga4
          3 client_e547b89c05043229 content_0e03de7680314cd5           1.000000       high refresh_and_review_ctr     visible_page|low_ctr|position_ok|low_ga4
          4 client_73cda7b4e4f265ea content_8e1334d6356668e3           1.000000       high refresh_and_review_ctr     visible_page|low_ctr|position_ok|low_ga4
          5 client_e547b89c05043229 content_8d7d99f109e19aa2           1.000000       high refresh_and_review_ctr visible_page|low_ctr|positio

## 4. Monitoring, escalation, and human review workflow

The review queue is only a prioritization layer. Its safe use is:

- inspect the highest-ranked items first
- verify the page against editorial context and business intent
- check whether low-visibility or low-engagement cases are genuinely worth a refresh
- treat model probability as a signal, not a final action decision

The audit logic from Week 6 still matters here: we do not claim causal certainty, only predictive support.

The final model remains the Week-5 Logistic Regression because Week 6's honest validation selected it over Random Forest under the assignment's rule.

In [11]:
if HAS_WAREHOUSE_ACCESS and not queue.empty:
    queue["review_flag"] = np.where(
        (queue["confidence"] == "low") | (queue["ga4_sessions_clean"] < 5) | (queue["gsc_impressions"] < 500),
        "manual_review",
        "standard_review",
    )
    review_summary = queue["review_flag"].value_counts().to_dict()
    print("Review queue summary:")
    print(review_summary)
else:
    print("No operational review summary generated because the warehouse-backed final queue was intentionally skipped in this environment.")

Review queue summary:
{'manual_review': 175757, 'standard_review': 684}


## 5. Export and reproduction

The final queue is saved only when the validated warehouse-backed pipeline is available. The notebook should never export the old starter reference queue or claim the starter model report is the final evidence.

In [12]:
output_path = WORK_OUTPUTS / "week7_final_queue.csv"
summary_path = WORK_OUTPUTS / "week7_queue_summary.json"

if HAS_WAREHOUSE_ACCESS and not queue.empty:
    queue_export = queue[[
        "final_rank",
        "client_hash_id",
        "content_hash_id",
        "model_probability",
        "confidence",
        "suggested_action",
        "reason_codes",
        "gsc_impressions",
        "gsc_clicks",
        "gsc_ctr",
        "gsc_avg_position_clean",
        "ga4_sessions_clean",
        "ga4_users_clean",
    ]].copy()
    queue_export.to_csv(output_path, index=False)
    summary = {
        "final_model": FINAL_MODEL_NAME,
        "target": "future_decline_proxy",
        "queue_rows": int(len(queue_export)),
        "confidence_counts": queue_export["confidence"].value_counts().to_dict(),
        "action_counts": queue_export["suggested_action"].value_counts().to_dict(),
        "feature_set": FEATURE_COLS,
        "provenance": "Week 5 decision snapshot + Week 6 honest validation + Week 7 operational queue",
    }
    summary_path.write_text(__import__("json").dumps(summary, indent=2), encoding="utf-8")
    print(f"Saved validated queue to: {output_path}")
    print(f"Saved provenance summary to: {summary_path}")
else:
    print("No queue export written because the environment is missing the FlyRank warehouse token and the final queue cannot be produced without the validated data path.")
    print("This is intentional and avoids reusing the stale starter-data artifacts.")

Saved validated queue to: /content/work/outputs/week7_final_queue.csv
Saved provenance summary to: /content/work/outputs/week7_queue_summary.json


## Self-check

- [x] Week 7 is now explicitly tied to the Week 5–6 validated pipeline rather than the starter reference artifacts.
- [x] The selected final model is Week-5 Logistic Regression, based on the actual Week 6 honest-split result.
- [x] The target remains `future_decline_proxy`, and the queue is built from decision-time features only.
- [x] The final queue generation is gated on the real warehouse-backed pipeline; it does not silently consume stale reference outputs.
- [x] The notebook is honest about when it cannot generate a queue without FlyRank warehouse access.